# TweetAge v5: modelos de lenguaje de caracteres y palabras

Conteos propios, Kneser-Ney interpolado (descuento absoluto 0.75), mezcla de log-verosimilitudes y sesgos por clase. Sin corpus externos ni modelos preentrenados. Solo se reutilizan la limpieza HTML, exclusion de textos defectuosos, split y metricas del equipo.

El 20 % queda reservado hasta terminar la seleccion en cinco pliegues del 80 %. Los scores de CV con sesgo se ajustan sobre predicciones fuera de muestra y no son una estimacion independiente: el holdout es la evaluacion independiente. No se envia nada a Kaggle.

In [1]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import time
from collections import Counter
from functools import lru_cache
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
CHAR_ORDERS = [3, 4, 5, 6]
WORD_ORDERS = [1, 2, 3]
LAMBDAS = [0.0, 0.25, 0.5, 0.75, 1.0]
DISCOUNT = 0.75
BASE_EPSILON = 1e-6
ORDER_LIMIT_SECONDS = 20 * 60
CLASSES = np.array(['13-17', '18-24', '25-34', '35-49', '50-64', '65-xx'])
BOS, EOS, UNK = '\x02', '\x03', '\x01'
roots = [Path.cwd(), Path.cwd() / 'workshops' / 'Kaggle-tweet-age', Path.cwd() / 'natural-language-processing-workshops' / 'workshops' / 'Kaggle-tweet-age']
OUT_DIR = next((candidate.resolve() for candidate in roots if (candidate / 'TweetAge_v5_charLM_kn.ipynb').is_file()), None)
assert OUT_DIR is not None, 'Ejecute desde la carpeta del notebook o la raiz del repositorio.'
NOTEBOOK_PATH = OUT_DIR / 'TweetAge_v5_charLM_kn.ipynb'
DATA_DIR = OUT_DIR / 'tweet-age-2026-puj'
SUB_PATH = OUT_DIR / 'age_submission_charLM_kn.csv'
SPLIT_PATH = OUT_DIR / 'split_idx_seed42.npz'
for output in [SUB_PATH, SPLIT_PATH]:
    if output.exists():
        raise FileExistsError(f'No se sobrescribe un archivo existente: {output}')
for name in ['age_train.csv', 'age_test.csv', 'age_submission.csv']:
    assert (DATA_DIR / name).is_file(), f'Falta el archivo {name}'
INPUT_HASHES = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() for name in ['age_train.csv', 'age_test.csv', 'age_submission.csv']}
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)
print('Semilla:', SEED, '| Salidas:', OUT_DIR)

Python 3.12.14 | numpy 2.5.4 | pandas 3.0.6 | sklearn 1.9.1
Semilla: 42 | Salidas: ./workshops/Kaggle-tweet-age


In [2]:
def check_compliance(notebook):
    forbidden = {'TfidfVectorizer', 'CountVectorizer', 'HashingVectorizer', 'DictVectorizer', 'transformers', 'embedding', 'embeddings', 'Word2Vec', 'GloVe'}
    problems = []
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        tree = ast.parse(source)
        for node in ast.walk(tree):
            names = []
            if isinstance(node, ast.Name):
                names.append(node.id)
            elif isinstance(node, ast.Attribute):
                names.append(node.attr)
            elif isinstance(node, ast.Import):
                names.extend(alias.name for alias in node.names)
            elif isinstance(node, ast.ImportFrom):
                names.append(node.module or '')
                names.extend(alias.name for alias in node.names)
            for name in names:
                if any(part.lower() in {value.lower() for value in forbidden} for part in name.split('.')) or name.startswith('sklearn.feature_extraction'):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica no permitida'))
    if problems:
        raise RuntimeError(f'Guardia de cumplimiento: codigo prohibido {problems}')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST superada: sin nombres/importaciones prohibidos en codigo; literales, comentarios y Markdown no se confunden con usos.')

Guardia AST superada: sin nombres/importaciones prohibidos en codigo; literales, comentarios y Markdown no se confunden con usos.


## Datos y normalizacion

Se comprueban BOM, separador y cabeceras: entrenamiento y prueba usan punto y coma y UTF-8 con BOM; la referencia usa coma sin BOM y solo se interpreta su cabecera. Los hashes verifican que los tres archivos permanezcan intactos.

Se decodifica HTML, se recortan extremos y se excluyen del entrenamiento textos vacios o con el artefacto de Excel `#¿NOMBRE?`. Despues se sustituyen URLs por `<url>` y menciones por `<user>`. Solo las repeticiones de letras identicas se limitan a tres; no se reducen signos ni emojis, ni se convierten mayusculas a minusculas. Las observaciones que quedan como `<url>` se cuentan, no se eliminan.

Se preserva el orden original y se divide despues de la misma limpieza que Kelly, antes de aprender vocabularios. El archivo NPZ contiene indices sobre el corpus limpio y sobre las filas originales, semilla y hash de entrada. No se agrupan duplicados para conservar la receta historica; esto puede favorecer ejemplos repetidos.

In [3]:
def verify_csv(path, expected_columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    actual_bom = prefix.startswith(b'\xef\xbb\xbf')
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    detected = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    columns = next(csv.reader([first_line], delimiter=detected))
    assert actual_bom == bom and detected == delimiter, (path.name, actual_bom, detected)
    assert columns == expected_columns, (path.name, columns)
    print(path.name, 'BOM=', actual_bom, 'separador=', repr(detected), 'cabecera=', columns)
    return columns

verify_csv(DATA_DIR / 'age_train.csv', ['id', 'text', 'age_range'], ';', True)
reference_columns = verify_csv(DATA_DIR / 'age_submission.csv', ['id', 'age_range'], ',', False)
train_raw = pd.read_csv(DATA_DIR / 'age_train.csv', sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert list(train_raw.columns) == ['id', 'text', 'age_range']
clean_text = train_raw['text'].fillna('').str.strip().map(html.unescape)
excluded = clean_text.eq('') | clean_text.str.contains('#\u00bfNOMBRE', regex=False)
train = train_raw.loc[~excluded].copy()
train['raw_row'] = np.flatnonzero(~excluded.to_numpy())
train['text'] = clean_text.loc[~excluded]
train = train.reset_index(drop=True)
assert set(train.age_range) == set(CLASSES), 'Se requieren exactamente los seis rangos originales.'
assert train.id.is_unique
print('Filas originales:', len(train_raw), '| Excluidas:', int(excluded.sum()), '| Utilizables:', len(train))
display(train.age_range.value_counts().reindex(CLASSES).rename('filas').to_frame())

URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)
MENTION_RE = re.compile(r'@\w+', re.UNICODE)
REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)
def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub('<url>', text)))

train['normalized'] = train.text.map(normalize_text)
print('Tuits solo URL tras normalizar:', int(train.normalized.str.strip().eq('<url>').sum()))
assert normalize_text('Hoooooola!!!! @Ana https://a.co') == 'Hooola!!!! <user> <url>'
assert normalize_text('JAJA \U0001f600\U0001f600\U0001f600\U0001f600 !!!!!') == 'JAJA \U0001f600\U0001f600\U0001f600\U0001f600 !!!!!'
idx_development, idx_holdout = train_test_split(np.arange(len(train)), test_size=0.2, stratify=train.age_range, random_state=SEED)
assert not np.intersect1d(idx_development, idx_holdout).size
with SPLIT_PATH.open('xb') as handle:
    np.savez_compressed(handle, train_idx=idx_development, valid_idx=idx_holdout, raw_train_idx=train.raw_row.to_numpy()[idx_development], raw_valid_idx=train.raw_row.to_numpy()[idx_holdout], retained_raw_idx=train.raw_row.to_numpy(), seed=np.array(SEED), train_sha256=np.array(INPUT_HASHES['age_train.csv']))
y_all = pd.Categorical(train.age_range, categories=CLASSES).codes.astype(np.int64)
texts_development = train.normalized.iloc[idx_development].to_numpy()
y_development = y_all[idx_development]
print('Desarrollo:', len(idx_development), '| Holdout reservado:', len(idx_holdout))
seen_texts = set(train.normalized.iloc[idx_development])
print('Holdout con texto normalizado presente en desarrollo:', int(train.normalized.iloc[idx_holdout].isin(seen_texts).sum()))

age_train.csv BOM= True separador= ';' cabecera= ['id', 'text', 'age_range']
age_submission.csv BOM= False separador= ',' cabecera= ['id', 'age_range']


Filas originales: 169336 | Excluidas: 155 | Utilizables: 169181


,filas
age_range,
13-17,28785
18-24,28626
25-34,28324
35-49,28226
50-64,28060
65-xx,27160


Tuits solo URL tras normalizar: 3785


Desarrollo: 135344 | Holdout reservado: 33837
Holdout con texto normalizado presente en desarrollo: 2561


## Kneser-Ney interpolado implementado a mano

Para historial h y siguiente simbolo w:

$$P(w|h)=\frac{\max(c(hw)-D,0)}{c(h)}+\frac{D N_{1+}(h*)}{c(h)} P(w|\mathrm{sufijo}(h)).$$

El orden superior usa frecuencias de ocurrencia; los inferiores usan el numero de predecesores distintos del n-grama, no su frecuencia bruta. La base unigramal es la distribucion de continuaciones de bigramas; el modelo de palabras de orden 1 tambien usa esa base. Para que simbolos ausentes en una clase tengan probabilidad positiva se agrega epsilon=1e-6 a cada conteo base del vocabulario de ajuste, incluido UNK, y se renormaliza.

Cada texto lleva n-1 BOS y un EOS; BOS nunca se predice. Los tokens se codifican reversiblemente con simbolos internos, sin vectores. El vocabulario compartido entre clases se aprende solo en el ajuste de cada pliegue: caracteres observados y palabras con al menos dos ocurrencias; las restantes pasan a UNK. El tokenizador conserva palabras Unicode, los marcadores `<url>`/`<user>` y signos/emojis separados.

Se suman log-probabilidades por texto, sin dividir por longitud:

$$s_c=\lambda\log P_{car}(texto|c)+(1-\lambda)\log P_{pal}(texto|c)+\log prior_c+b_c.$$

Los priors solo usan etiquetas de ajuste. Se reutilizan conteos entre ordenes, nunca entre pliegues; los tiempos de construccion compartidos se reparten entre ordenes y tambien se muestran por separado. Los pesos lambda no requieren reentrenar. Las pruebas comprueban continuaciones, suma de probabilidades, marcadores y desconocidos.

In [4]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text)

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    vocabulary = sorted(token for token, count in frequencies.items() if count >= cutoff)
    return {token: chr(index + 256) for index, token in enumerate(vocabulary)}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

class InterpolatedKN:
    def __init__(self, max_order, support, discount=DISCOUNT):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}
        self.discount = discount

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        padding = BOS * (highest - 1)
        for sequence in sequences:
            padded = padding + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        def histories(counts):
            totals, types = Counter(), Counter()
            for gram, count in counts.items():
                totals[gram[:-1]] += count
                types[gram[:-1]] += 1
            return {history: (total, self.discount * types[history]) for history, total in totals.items()}
        self.raw_hist = {order: histories(counts) for order, counts in self.raw.items()}
        self.cont_hist = {order: histories(counts) for order, counts in self.continuation.items() if order > 1}
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0))
        if not total:
            return lower
        return (max(self.continuation[order].get(gram, 0) - self.discount, 0) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0))
        if not total:
            return lower
        return (max(self.raw[order].get(gram, 0) - self.discount, 0) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-12:
            raise ArithmeticError(('Probabilidad invalida', repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()


def test_kn():
    model = InterpolatedKN(3, set('abc')).fit(['ab', 'ab', 'ac', 'bc'])
    assert model.raw[2]['ab'] == 2
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 3
    for order in [1, 2, 3]:
        for context in ['', BOS, BOS * 2, 'a', 'b', 'zz']:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            total = sum(model.probability(history + symbol) for symbol in model.support)
            assert abs(total - 1) < 1e-10, (order, repr(history), total)
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 3) > model.log_probability('cc', 3)
    model.close()
    print('Pruebas KN: continuaciones distintas, normalizacion, BOS/EOS y UNK correctos.')

test_kn()

Pruebas KN: continuaciones distintas, normalizacion, BOS/EOS y UNK correctos.


In [5]:
timing_rows = []
reductions = []

def fit_score_level(fit_texts, fit_labels, app_texts, level, orders, phase, fold):
    started = time.perf_counter()
    mapping = make_encoder(fit_texts, level)
    encoded_fit = [encode_text(text, level, mapping) for text in fit_texts]
    encoded_app = [encode_text(text, level, mapping) for text in app_texts]
    preprocessing = time.perf_counter() - started
    matrices = {order: np.zeros((len(app_texts), len(CLASSES)), dtype=np.float64) for order in orders}
    fit_seconds = 0.0
    score_seconds = {order: 0.0 for order in orders}
    support = set(mapping.values())
    for class_index, label in enumerate(CLASSES):
        class_started = time.perf_counter()
        sequences = (sequence for sequence, target in zip(encoded_fit, fit_labels) if target == class_index)
        model = InterpolatedKN(max(orders), support).fit(sequences)
        fit_seconds += time.perf_counter() - class_started
        for order in orders:
            score_started = time.perf_counter()
            matrices[order][:, class_index] = [model.log_probability(sequence, order) for sequence in encoded_app]
            score_seconds[order] += time.perf_counter() - score_started
        model.close()
        del model
        gc.collect()
        print(phase, fold, level, label, 'listo', round(time.perf_counter() - started, 1), 's', flush=True)
    times = {}
    for order in orders:
        times[order] = score_seconds[order] + (fit_seconds + preprocessing) / len(orders)
        timing_rows.append(dict(fase=phase, pliegue=fold, nivel=level, orden=order, ajuste_compartido_s=fit_seconds + preprocessing, puntuacion_s=score_seconds[order], segundos_amortizados=times[order]))
    return matrices, times

def macro_fast(targets, predictions):
    matrix = np.bincount(targets * len(CLASSES) + predictions, minlength=len(CLASSES) ** 2).reshape(len(CLASSES), len(CLASSES))
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(2 * matrix.diagonal(), denominator, out=np.zeros(len(CLASSES), dtype=float), where=denominator != 0).mean())

def calibrate_bias(scores, targets):
    bias = np.zeros(len(CLASSES), dtype=float)
    baseline = macro_fast(targets, scores.argmax(axis=1))
    current = baseline
    sorted_scores = np.sort(scores, axis=1)
    scale = max(0.5, min(20.0, float(np.median(sorted_scores[:, -1] - sorted_scores[:, -2]))))
    for step in [scale * 2, scale, scale / 2, scale / 4]:
        for class_index in range(len(CLASSES)):
            adjusted = scores + bias
            adjusted[:, class_index] = -np.inf
            alternatives = adjusted.argmax(axis=1)
            alternative_max = adjusted[np.arange(len(targets)), alternatives]
            best_value = bias[class_index]
            for value in bias[class_index] + step * np.array([-2, -1, -0.5, 0, 0.5, 1, 2]):
                class_score = scores[:, class_index] + value
                choose = (class_score > alternative_max) | ((class_score == alternative_max) & (class_index < alternatives))
                predictions = np.where(choose, class_index, alternatives)
                result = macro_fast(targets, predictions)
                if result > current + 1e-12:
                    current, best_value = result, float(value)
            bias[class_index] = best_value
        bias -= bias.mean()
    return bias, baseline, macro_fast(targets, (scores + bias).argmax(axis=1))

_test_targets = np.array([0, 1, 2, 3, 4, 5, 5])
_test_predictions = np.array([0, 0, 2, 3, 5, 5, 4])
assert abs(macro_fast(_test_targets, _test_predictions) - f1_score(_test_targets, _test_predictions, labels=np.arange(6), average='macro', zero_division=0)) < 1e-12
_test_scores = np.random.default_rng(SEED).normal(size=(len(_test_targets), 6))
_test_bias, _before, _after = calibrate_bias(_test_scores, _test_targets)
assert _after >= _before and abs(_test_bias.mean()) < 1e-10
print('Metricas y calibracion por coordenadas verificadas.')

Metricas y calibracion por coordenadas verificadas.


## Piloto, seleccion y sesgos

El piloto usa una muestra estratificada del 10 % de desarrollo, nunca el holdout, y la divide internamente 80/20 para medir todos los ordenes. Si el tiempo medido/amortizado de un orden supera 1200 s, se elimina ese orden y se informa; se conserva al menos el mas barato. En CV tambien se excluyen de la seleccion los ordenes que exceden el limite. Las proyecciones escaladas del piloto son orientativas, no mediciones completas.

Cinco pliegues estratificados, semilla 42, producen predicciones fuera de muestra de todo el 80 %. Se prueban todos los pares de ordenes y cinco pesos. Para cada configuracion se ajusta b por coordenadas sobre esas predicciones OOF: cuatro pasadas con escalas decrecientes, siete desplazamientos por clase y aceptacion solo si aumenta F1 macro. Se centra b para identificarlo. Es una busqueda local, no garantiza un maximo global. Se informan F1 antes/despues y media/desviacion entre pliegues.

El F1 CV calibrado es un indicador de ajuste, optimista por reutilizar las etiquetas OOF para b y la seleccion; el holdout permanece independiente. En empates se prefieren ordenes menores y luego lambda menor. El tiempo de componentes se comparte entre filas: no se suman los tiempos de todas las combinaciones como si fueran entrenamientos independientes.

In [6]:
pilot_idx, _ = train_test_split(np.arange(len(y_development)), train_size=0.1, stratify=y_development, random_state=SEED)
pilot_fit, pilot_app = train_test_split(pilot_idx, test_size=0.2, stratify=y_development[pilot_idx], random_state=SEED)
active_orders = {}
for level, orders in [('char', CHAR_ORDERS), ('word', WORD_ORDERS)]:
    pilot_scores, measured = fit_score_level(texts_development[pilot_fit], y_development[pilot_fit], texts_development[pilot_app], level, orders, 'piloto_10pct', 0)
    active = [order for order in orders if measured[order] <= ORDER_LIMIT_SECONDS]
    if not active:
        active = [min(measured, key=measured.get)]
    for order in set(orders) - set(active):
        reductions.append(f'Piloto: {level} orden {order} excluido por {measured[order]:.1f} s > 1200 s.')
    active_orders[level] = active
    del pilot_scores
    gc.collect()
pilot_table = pd.DataFrame(timing_rows)
pilot_table['estimacion_pliegue_completo_s'] = pilot_table.segundos_amortizados * 10
display(pilot_table)
print('Ordenes activos:', active_orders, '| Reducciones:', reductions or 'ninguna')

piloto_10pct 0 char 13-17 listo 1.0 s


piloto_10pct 0 char 18-24 listo 1.7 s


piloto_10pct 0 char 25-34 listo 2.5 s


piloto_10pct 0 char 35-49 listo 3.2 s


piloto_10pct 0 char 50-64 listo 3.9 s


...

piloto_10pct 0 word 35-49 listo 0.5 s


piloto_10pct 0 word 50-64 listo 0.6 s


piloto_10pct 0 word 65-xx listo 0.7 s


,fase,pliegue,nivel,orden,ajuste_compartido_s,puntuacion_s,segundos_amortizados,estimacion_pliegue_completo_s
0,piloto_10pct,0,char,3,1.244794,0.241510,0.552709,5.527088
1,piloto_10pct,0,char,4,1.244794,0.523684,0.834882,8.348821
2,piloto_10pct,0,char,5,1.244794,0.990480,1.301678,13.016782
3,piloto_10pct,0,char,6,1.244794,1.394184,1.705382,17.053822
4,piloto_10pct,0,word,1,0.197819,0.047541,0.113481,1.134808
5,piloto_10pct,0,word,2,0.197819,0.104455,0.170394,1.703942
6,piloto_10pct,0,word,3,0.197819,0.203614,0.269553,2.695534


Ordenes activos: {'char': [3, 4, 5, 6], 'word': [1, 2, 3]} | Reducciones: ninguna


In [7]:
cv_started = time.perf_counter()
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(texts_development, y_development))
oof_char = {order: np.zeros((len(y_development), 6)) for order in active_orders['char']}
oof_word = {order: np.zeros((len(y_development), 6)) for order in active_orders['word']}
oof_prior = np.zeros((len(y_development), 6))
component_times = {'char': Counter(), 'word': Counter()}
slow_orders = {'char': set(), 'word': set()}
for fold_number, (fit_idx, app_idx) in enumerate(folds, 1):
    counts = np.bincount(y_development[fit_idx], minlength=6)
    oof_prior[app_idx] = np.log(counts / counts.sum())
    for level, storage in [('char', oof_char), ('word', oof_word)]:
        scores, measured = fit_score_level(texts_development[fit_idx], y_development[fit_idx], texts_development[app_idx], level, active_orders[level], 'CV', fold_number)
        for order in active_orders[level]:
            storage[order][app_idx] = scores[order]
            component_times[level][order] += measured[order]
            if measured[order] > ORDER_LIMIT_SECONDS:
                slow_orders[level].add(order)
        del scores
        gc.collect()
    print('Pliegue', fold_number, 'de 5 terminado; tiempo CV', round(time.perf_counter() - cv_started, 1), 's', flush=True)
for level in ['char', 'word']:
    acceptable = [order for order in active_orders[level] if order not in slow_orders[level]]
    if slow_orders[level]:
        reductions.append(f'CV: {level} ordenes {sorted(slow_orders[level])} superaron 1200 s por pliegue; se excluyen salvo que no quede alternativa.')
    active_orders[level] = acceptable or [min(active_orders[level], key=lambda order: component_times[level][order])]

search_rows = []
bias_vectors = {}
for char_order, word_order, weight in product(active_orders['char'], active_orders['word'], LAMBDAS):
    calibration_started = time.perf_counter()
    scores = weight * oof_char[char_order] + (1 - weight) * oof_word[word_order] + oof_prior
    bias, raw_f1, calibrated_f1 = calibrate_bias(scores, y_development)
    predictions = (scores + bias).argmax(axis=1)
    fold_f1 = [macro_fast(y_development[app_idx], predictions[app_idx]) for _, app_idx in folds]
    calibration_seconds = time.perf_counter() - calibration_started
    key = (char_order, word_order, weight)
    bias_vectors[key] = bias
    search_rows.append(dict(orden_car=char_order, orden_pal=word_order, lambda_car=weight, f1_macro_sin_sesgo=raw_f1, f1_macro_cv=calibrated_f1, f1_pliegues_media=float(np.mean(fold_f1)), f1_pliegues_std=float(np.std(fold_f1)), componentes_cv_s=component_times['char'][char_order] + component_times['word'][word_order], calibracion_s=calibration_seconds))
    print('Configuracion', key, 'F1 OOF ajustado', round(calibrated_f1, 5), 'calibracion', round(calibration_seconds, 2), 's', flush=True)
results = pd.DataFrame(search_rows).sort_values(['f1_macro_cv', 'orden_car', 'orden_pal', 'lambda_car'], ascending=[False, True, True, True]).reset_index(drop=True)
best = results.iloc[0]
best_char, best_word, best_weight = int(best.orden_car), int(best.orden_pal), float(best.lambda_car)
best_bias = bias_vectors[(best_char, best_word, best_weight)].copy()
print('Tabla completa (tiempo de componentes reutilizado entre pesos; no sumar entre filas):')
print(results.sort_values(['orden_car', 'orden_pal', 'lambda_car']).to_string(index=False))
print('Mejor configuracion:', best_char, best_word, best_weight, '| Sesgos:', dict(zip(CLASSES, best_bias)))
print('Tiempo CV y seleccion:', round(time.perf_counter() - cv_started, 1), 's')
print('Reducciones:', reductions or 'ninguna')
display(pd.DataFrame(timing_rows))

CV 1 char 13-17 listo 7.9 s


CV 1 char 18-24 listo 14.8 s


CV 1 char 25-34 listo 22.0 s


CV 1 char 35-49 listo 29.4 s


CV 1 char 50-64 listo 36.6 s


...

Configuracion (6, 3, 0.5) F1 OOF ajustado 0.43003 calibracion 0.16 s


Configuracion (6, 3, 0.75) F1 OOF ajustado 0.43442 calibracion 0.16 s


Configuracion (6, 3, 1.0) F1 OOF ajustado 0.42747 calibracion 0.16 s


Tabla completa (tiempo de componentes reutilizado entre pesos; no sumar entre filas):
 orden_car  orden_pal  lambda_car  f1_macro_sin_sesgo  f1_macro_cv  f1_pliegues_media  f1_pliegues_std  componentes_cv_s  calibracion_s
         3          1        0.00            0.354626     0.367190           0.367182         0.001361         31.632725       0.156992
         3          1        0.25            0.386333     0.394776           0.394767         0.001708         31.632725       0.155689
         3          1        0.50            0.399912     0.402776           0.402763         0.002087         31.632725       0.155576
         3          1        0.75            0.406686     0.409943           0.409927         0.002169         31.632725       0.154674
         3          1        1.00            0.394946     0.399488           0.399481         0.001881         31.632725       0.152378
         3          2        0.00            0.397974     0.399747           0.399734         0.00

,fase,pliegue,nivel,orden,ajuste_compartido_s,puntuacion_s,segundos_amortizados
0,piloto_10pct,0,char,3,1.244794,0.241510,0.552709
1,piloto_10pct,0,char,4,1.244794,0.523684,0.834882
2,piloto_10pct,0,char,5,1.244794,0.990480,1.301678
3,piloto_10pct,0,char,6,1.244794,1.394184,1.705382
4,piloto_10pct,0,word,1,0.197819,0.047541,0.113481
5,piloto_10pct,0,word,2,0.197819,0.104455,0.170394
6,piloto_10pct,0,word,3,0.197819,0.203614,0.269553
7,CV,1,char,3,11.012066,1.978298,4.731314
8,CV,1,char,4,11.012066,4.661709,7.414726
9,CV,1,char,5,11.012066,9.018128,11.771145


## Evaluacion unica del holdout

Se congelan ordenes, lambda y sesgos antes de evaluar. Referencias historicas de validacion: v2=0.5810 y v3=0.6075 (script v3, commit a52fd19; el notebook 13a1f22 resume v3 como 0.607). No se comparan scores de Kaggle.

Se reproduce la receta del split: mismo CSV limpio, orden, 80/20 y semilla 42. Al no disponer de indices historicos de Kelly no se puede demostrar identidad documental; deberia coincidir si el archivo y la implementacion de sklearn coinciden. La normalizacion nueva es diferente, pero no cambia las filas del split. Los duplicados siguen siendo un riesgo compartido. Las interpretaciones de los diez errores son hipotesis basadas en el texto, no causas demostradas.

In [8]:
assert 'holdout_f1' not in globals(), 'El holdout ya se evaluo en este kernel; no se vuelve a usar para seleccionar.'
holdout_started = time.perf_counter()
texts_holdout = train.normalized.iloc[idx_holdout].to_numpy()
char_holdout, _ = fit_score_level(texts_development, y_development, texts_holdout, 'char', [best_char], 'holdout', 0)
word_holdout, _ = fit_score_level(texts_development, y_development, texts_holdout, 'word', [best_word], 'holdout', 0)
prior_development = np.bincount(y_development, minlength=6) / len(y_development)
holdout_scores = best_weight * char_holdout[best_char] + (1 - best_weight) * word_holdout[best_word] + np.log(prior_development) + best_bias
holdout_pred = holdout_scores.argmax(axis=1)
y_holdout = y_all[idx_holdout]
holdout_f1 = f1_score(y_holdout, holdout_pred, labels=np.arange(6), average='macro', zero_division=0)
report = classification_report(y_holdout, holdout_pred, labels=np.arange(6), target_names=CLASSES, digits=4, zero_division=0, output_dict=True)
class_table = pd.DataFrame(report).T.loc[CLASSES, ['precision', 'recall', 'f1-score', 'support']]
print('F1 macro holdout:', holdout_f1)
display(class_table)
display(pd.DataFrame(confusion_matrix(y_holdout, holdout_pred, labels=np.arange(6)), index=pd.Index(CLASSES, name='real'), columns=pd.Index(CLASSES, name='predicha')))
comparison = pd.DataFrame({'modelo': ['v2 equipo', 'v3 equipo', 'v5 KN'], 'f1_macro_holdout': [0.5810, 0.6075, holdout_f1]})
comparison['diferencia_v5_menos_referencia'] = holdout_f1 - comparison.f1_macro_holdout
display(comparison)

def interpret_error(text, real, predicted):
    if text.strip() == '<url>':
        return 'Solo queda una URL: no hay contenido lexico para distinguir edades.'
    if len(TOKEN_RE.findall(text)) < 5:
        return 'Texto muy corto: pocas pistas linguisticas; explicacion tentativa.'
    if abs(real - predicted) == 1:
        return 'Rangos contiguos con vocabulario y estilo posiblemente compartidos.'
    if '<user>' in text:
        return 'La normalizacion oculta la identidad mencionada; posible perdida de una pista.'
    return 'El contenido puede ser compartido por edades distintas; requiere revision humana.'

error_positions = np.flatnonzero(y_holdout != holdout_pred)[:10]
error_table = pd.DataFrame([{'texto': train.text.iloc[idx_holdout[position]], 'real': CLASSES[y_holdout[position]], 'predicha': CLASSES[holdout_pred[position]], 'interpretacion': interpret_error(texts_holdout[position], int(y_holdout[position]), int(holdout_pred[position]))} for position in error_positions])
with pd.option_context('display.max_colwidth', 180):
    display(error_table)
print('Tiempo holdout:', round(time.perf_counter() - holdout_started, 1), 's')
del char_holdout, word_holdout
gc.collect()

holdout 0 char 13-17 listo 2.0 s


holdout 0 char 18-24 listo 3.4 s


holdout 0 char 25-34 listo 4.8 s


holdout 0 char 35-49 listo 6.4 s


holdout 0 char 50-64 listo 8.0 s


...

predicha,13-17,18-24,25-34,35-49,50-64,65-xx
real,,,,,,
13-17,2615,1494,740,522,262,124
18-24,1048,2297,1154,664,352,211
25-34,518,1170,2069,1031,531,346
35-49,247,616,1028,2371,827,556
50-64,116,299,595,1001,2868,733
65-xx,134,269,383,870,921,2855


,modelo,f1_macro_holdout,diferencia_v5_menos_referencia
0,v2 equipo,0.581000,-0.131552
1,v3 equipo,0.607500,-0.158052
2,v5 KN,0.449448,0.000000


,texto,real,predicha,interpretacion
0,Santa marta me espera...inicio fabricacion taller alfering proyecto maceo,35-49,50-64,Rangos contiguos con vocabulario y estilo posiblemente compartidos.
1,A por el 3-0 #VamosColombia,13-17,25-34,El contenido puede ser compartido por edades distintas; requiere revision humana.
2,Me ha gustado un video de @YouTube (https://t.co/svJZP58w2w - Martina La Peligrosa - Me Voy (Video Oficial)).,18-24,35-49,La normalizacion oculta la identidad mencionada; posible perdida de una pista.
3,https://t.co/zwrmSUeG3D,65-xx,35-49,Solo queda una URL: no hay contenido lexico para distinguir edades.
4,"I loved this book when I was a child. In fact, I think I learned English thanks to it. My sister read http://t.co/Qb5P6Wpewo",25-34,18-24,Rangos contiguos con vocabulario y estilo posiblemente compartidos.
5,RT @OfficialNinel: #SiLaVidaFueraJusta El Toro cornaria al torero!! http://t.co/V8kFAcH9,18-24,35-49,La normalizacion oculta la identidad mencionada; posible perdida de una pista.
6,RT @marcalvarezg: https://t.co/mN47wDIQRa,35-49,13-17,Texto muy corto: pocas pistas linguisticas; explicacion tentativa.
7,http://t.co/NQpgQ74Xsi,65-xx,35-49,Solo queda una URL: no hay contenido lexico para distinguir edades.
8,Sin palabras tambien se llenan paginas,50-64,25-34,El contenido puede ser compartido por edades distintas; requiere revision humana.
9,RT @miiguelangel92: Como les gusta CON/SI. proteccion https://t.co/b22rVbsYwz,25-34,35-49,Rangos contiguos con vocabulario y estilo posiblemente compartidos.


Tiempo holdout: 17.8 s


0

In [9]:
final_started = time.perf_counter()
verify_csv(DATA_DIR / 'age_test.csv', ['id', 'text'], ';', True)
test = pd.read_csv(DATA_DIR / 'age_test.csv', sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert list(test.columns) == ['id', 'text']
assert test.id.is_unique and test.id.ne('').all()
test_clean = test.text.fillna('').str.strip().map(html.unescape)
test_normalized = test_clean.map(normalize_text).to_numpy()
print('Test:', len(test), '| Solo URL:', int(np.sum(np.char.strip(test_normalized.astype(str)) == '<url>')))
char_final, _ = fit_score_level(train.normalized.to_numpy(), y_all, test_normalized, 'char', [best_char], 'final', 0)
word_final, _ = fit_score_level(train.normalized.to_numpy(), y_all, test_normalized, 'word', [best_word], 'final', 0)
prior_final = np.bincount(y_all, minlength=6) / len(y_all)
final_scores = best_weight * char_final[best_char] + (1 - best_weight) * word_final[best_word] + np.log(prior_final) + best_bias
submission = pd.DataFrame({'id': test.id, 'age_range': CLASSES[final_scores.argmax(axis=1)]})
assert list(submission.columns) == reference_columns == ['id', 'age_range']
assert len(submission) == len(test) and submission.id.is_unique
assert submission.age_range.isin(CLASSES).all() and submission.id.tolist() == test.id.tolist()
with SUB_PATH.open('x', encoding='utf-8', newline='') as handle:
    submission.to_csv(handle, sep=',', index=False)
with SUB_PATH.open('rb') as handle:
    first_bytes = handle.read(64)
assert not first_bytes.startswith(b'\xef\xbb\xbf')
assert first_bytes.splitlines()[0] == b'id,age_range'
assert INPUT_HASHES == {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() for name in INPUT_HASHES}, 'Se alteraron los datos de entrada.'
print('Filas test/submission:', len(test), len(submission), '| Coinciden:', len(test) == len(submission))
print('Submission:', SUB_PATH)
print('Split:', SPLIT_PATH)
print('Notebook:', NOTEBOOK_PATH)
print('Tiempo final:', round(time.perf_counter() - final_started, 1), 's')
display(submission.head())
display(submission.age_range.value_counts().reindex(CLASSES).rename('predicciones').to_frame())
display(pd.DataFrame(timing_rows))

age_test.csv BOM= True separador= ';' cabecera= ['id', 'text']


Test: 112892 | Solo URL: 2438


final 0 char 13-17 listo 4.2 s


final 0 char 18-24 listo 7.5 s


final 0 char 25-34 listo 10.8 s


...

final 0 word 65-xx listo 23.1 s


Filas test/submission: 112892 112892 | Coinciden: True
Submission: ./workshops/Kaggle-tweet-age/age_submission_charLM_kn.csv
Split: ./workshops/Kaggle-tweet-age/split_idx_seed42.npz
Notebook: ./workshops/Kaggle-tweet-age/TweetAge_v5_charLM_kn.ipynb
Tiempo final: 45.2 s


,id,age_range
0,1,65-xx
1,2,18-24
2,3,13-17
3,4,65-xx
4,5,65-xx


,predicciones
age_range,
13-17,15476
18-24,20568
25-34,19980
35-49,21707
50-64,19120
65-xx,16041


,fase,pliegue,nivel,orden,ajuste_compartido_s,puntuacion_s,segundos_amortizados
0,piloto_10pct,0,char,3,1.244794,0.241510,0.552709
1,piloto_10pct,0,char,4,1.244794,0.523684,0.834882
2,piloto_10pct,0,char,5,1.244794,0.990480,1.301678
3,piloto_10pct,0,char,6,1.244794,1.394184,1.705382
4,piloto_10pct,0,word,1,0.197819,0.047541,0.113481
5,piloto_10pct,0,word,2,0.197819,0.104455,0.170394
6,piloto_10pct,0,word,3,0.197819,0.203614,0.269553
7,CV,1,char,3,11.012066,1.978298,4.731314
8,CV,1,char,4,11.012066,4.661709,7.414726
9,CV,1,char,5,11.012066,9.018128,11.771145


In [10]:
worst_class = class_table['f1-score'].idxmin()
delta_v3 = holdout_f1 - 0.6075
summary_text = f'''## Resumen de la corrida

Mejor orden de caracteres: **{best_char}**; palabras: **{best_word}**; peso de caracteres: **{best_weight:.2f}**; descuento: **{DISCOUNT}**. F1 macro CV ajustado: **{float(best.f1_macro_cv):.5f}**; sin sesgo: **{float(best.f1_macro_sin_sesgo):.5f}**. El ajuste OOF mejora el indicador de desarrollo, pero no equivale a una mejora independiente.

F1 macro en el holdout reservado: **{holdout_f1:.5f}**. Diferencia frente a v2: **{holdout_f1 - 0.5810:+.5f}**; frente a v3: **{delta_v3:+.5f}**. Se reproduce la receta del split historico; no hay indices historicos para demostrar identidad exacta. Son modelos distintos y no se han reejecutado las referencias.

La clase con peor F1 es **{worst_class}**, con **{class_table.loc[worst_class, 'f1-score']:.5f}**. Posiblemente comparte expresiones con otras edades o aporta pocas pistas en un solo tuit; es una hipotesis, no una causa demostrada. La normalizacion de cuentas y URLs tambien elimina pistas particulares.

Antes de enviar: revisar con el equipo las reglas, el rendimiento por clase, los errores y el cupo disponible; decidir si esta alternativa legal merece uno de los envios. No confundir cumplimiento con superioridad predictiva. No se envio nada a Kaggle.

Filas de prueba y salida: **{len(test)} / {len(submission)}**, iguales. Reducciones de rejilla: **{'; '.join(reductions) if reductions else 'ninguna'}**.

Sesgos por clase: `{dict(zip(CLASSES.tolist(), best_bias.tolist()))}`.
'''
print(results.sort_values(['orden_car', 'orden_pal', 'lambda_car']).to_string(index=False))
display(Markdown(summary_text))

 orden_car  orden_pal  lambda_car  f1_macro_sin_sesgo  f1_macro_cv  f1_pliegues_media  f1_pliegues_std  componentes_cv_s  calibracion_s
         3          1        0.00            0.354626     0.367190           0.367182         0.001361         31.632725       0.156992
         3          1        0.25            0.386333     0.394776           0.394767         0.001708         31.632725       0.155689
         3          1        0.50            0.399912     0.402776           0.402763         0.002087         31.632725       0.155576
         3          1        0.75            0.406686     0.409943           0.409927         0.002169         31.632725       0.154674
         3          1        1.00            0.394946     0.399488           0.399481         0.001881         31.632725       0.152378
         3          2        0.00            0.397974     0.399747           0.399734         0.003038         37.506823       0.157347
         3          2        0.25            0.4

## Resumen de la corrida

Mejor orden de caracteres: **4**; palabras: **3**; peso de caracteres: **0.75**; descuento: **0.75**. F1 macro CV ajustado: **0.43847**; sin sesgo: **0.43539**. El ajuste OOF mejora el indicador de desarrollo, pero no equivale a una mejora independiente.

F1 macro en el holdout reservado: **0.44945**. Diferencia frente a v2: **-0.13155**; frente a v3: **-0.15805**. Se reproduce la receta del split historico; no hay indices historicos para demostrar identidad exacta. Son modelos distintos y no se han reejecutado las referencias.

La clase con peor F1 es **25-34**, con **0.35568**. Posiblemente comparte expresiones con otras edades o aporta pocas pistas en un solo tuit; es una hipotesis, no una causa demostrada. La normalizacion de cuentas y URLs tambien elimina pistas particulares.

Antes de enviar: revisar con el equipo las reglas, el rendimiento por clase, los errores y el cupo disponible; decidir si esta alternativa legal merece uno de los envios. No confundir cumplimiento con superioridad predictiva. No se envio nada a Kaggle.

Filas de prueba y salida: **112892 / 112892**, iguales. Reducciones de rejilla: **ninguna**.

Sesgos por clase: `{'13-17': -0.16189050372754862, '18-24': 0.16189050372754868, '25-34': 0.4856715111826459, '35-49': 0.485671511182646, '50-64': -0.16189050372754862, '65-xx': -0.8094525186377433}`.


## Resumen final de la corrida

Se probaron las **60 combinaciones**, sin reducir la rejilla: caracteres de orden 3 a 6, palabras de orden 1 a 3 y cinco pesos. La mejor fue **caracteres 4 + palabras 3, lambda=0.75, D=0.75**. El sesgo por clase aumento el F1 OOF de ajuste de **0.43539 a 0.43847**; esa mejora usa las etiquetas de desarrollo para calibrar y no es una ganancia independiente. La CV y seleccion tardaron **276.5 s**, el holdout **17.8 s** y el entrenamiento/prediccion final **45.2 s**. Los tiempos de cada orden y configuracion estan en las salidas anteriores.

En el holdout reservado, el **F1 macro fue 0.44945**. No mejoro las referencias publicadas: **-0.13155 frente a v2 (0.5810)** y **-0.15805 frente a v3 (0.6075)**. Se reproduce la receta de la particion historica (limpieza, orden de filas, split estratificado 80/20, semilla 42), pero no hay indices historicos para demostrar identidad exacta ni se reejecutaron esas versiones. La normalizacion y el modelo son distintos.

| Edad | F1 holdout |
|---|---:|
| 13-17 | 0.501198 |
| 18-24 | 0.386994 |
| 25-34 | 0.355682 |
| 35-49 | 0.391771 |
| 50-64 | 0.504352 |
| 65-xx | 0.556693 |

La clase **25-34** fue la mas dificil. Su confusion con rangos vecinos sugiere vocabulario y estilos compartidos; es una hipotesis, no una causa probada. Reemplazar cuentas y URLs tambien elimina pistas particulares. Se conservaron **3785** tuits de entrenamiento que quedan solo como `<url>`. Hay textos normalizados repetidos entre desarrollo y holdout, un riesgo del split historico que se reporta en la celda de datos.

Sesgos, en orden de edad: **[-0.1618905, 0.1618905, 0.4856715, 0.4856715, -0.1618905, -0.8094525]**. El vocabulario y los conteos se ajustaron desde cero en cada pliegue, sin utilizar el test para seleccionar parametros. Los nombres prohibidos no aparecen como usos de codigo; la guardia AST y las pruebas de Kneser-Ney pasaron.

El entrenamiento final uso las **169181 filas limpias** del train. La prueba y la salida tienen **112892 filas cada una**, con los mismos IDs en el mismo orden, IDs unicos y solo clases validas. La salida usa coma, UTF-8 sin BOM y cabecera `id,age_range`. Los hashes confirman que los archivos preexistentes permanecen intactos.

**Decision antes de Kaggle:** no presentar este resultado como una mejora ni gastar un envio solo por superar la guardia de cumplimiento. Revisar las reglas y los errores con el equipo y decidir si interesa registrar una linea base permitida, dado que rinde menos que las versiones publicadas con representaciones prohibidas. No se envio nada a Kaggle ni se hicieron commits o push.